In [32]:
# imports
import pandas as pd
import re
import html

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

In [33]:
bills = pd.read_csv("bill_details.csv")
# rename columns
rename = {'Legislation Number':'bill_id',  'Title':'bill_name',
       'Latest Tracker Stage':'status', 'Latest Summary':'summary'}
bills = bills.rename(columns=rename).drop('Congress', axis=1)

# keep bills that have a status related to House or Senate
bills = bills.loc[bills['status'].isin(['Became Law', 'Passed House', 'Resolving Differences', 'Passed Senate', 'Failed Senate'])]
bills = bills.head() #TODO DELETE

In [34]:
# clean bill summaries
def clean_html_text(text):
    # remove <tags> and other HTML characters
    return html.unescape(re.sub(r'<.*?>', '', text))

bills['summary'] = bills['summary'].apply(clean_html_text)
bills.head()

,bill_id,URL,bill_name,status,summary
0,H.R. 5371,https://www.congress.gov/bill/119th-congress/h...,"Continuing Appropriations, Agriculture, Legisl...",Became Law,"Continuing Appropriations, Agriculture, Legisl..."
1,H.R. 5143,https://www.congress.gov/bill/119th-congress/h...,District of Columbia Policing Protection Act o...,Passed House,District of Columbia Policing Protection ActTh...
2,H.R. 5140,https://www.congress.gov/bill/119th-congress/h...,To lower the age at which a minor may be tried...,Passed House,This bill lowers the age at which an individua...
3,H.R. 5125,https://www.congress.gov/bill/119th-congress/h...,District of Columbia Judicial Nominations Refo...,Passed House,District of Columbia Judicial Nominations Refo...
4,H.R. 5100,https://www.congress.gov/bill/119th-congress/h...,"To extend the SBIR and STTR programs, and for ...",Passed House,This bill reauthorizes through FY2026 the Smal...


In [35]:
# tokenize summaries
tokenizer = Tokenizer(num_words=10000)  # keep top 10k words
tokenizer.fit_on_texts(bills['summary'])
sequences = tokenizer.texts_to_sequences(bills['summary'])

maxlen = 500  # max number of tokens per summary
X = pad_sequences(sequences, maxlen=maxlen)

# one hot encode labels
le = LabelEncoder()
y_int = le.fit_transform(bills['status'])
y = to_categorical(y_int)